# Notebook 3: The Heston Stochastic Volatility Model

Black-Scholes prices all options with a single constant volatility, but real option
markets show a "volatility smile" — different implied vols at different strikes.
This notebook introduces the Heston model, which treats volatility itself as a
stochastic process, and shows how it generates a realistic skew. The parameter
sensitivity section is useful for building intuition about what each Heston input
actually controls.

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(),
    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'build'))

import mc_engine
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
import math

def bs_call(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return float(S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2))

def implied_vol_nr(price, S, K, r, T, sigma0=0.20, max_iter=100, tol=1e-8):
    """Newton-Raphson implied volatility inversion. Returns None on failure."""
    sigma = sigma0
    for _ in range(max_iter):
        c  = bs_call(S, K, r, sigma, T)
        d1 = (math.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
        vega = S * math.sqrt(T) * norm.pdf(d1)
        if vega < 1e-12:
            return None
        sigma -= (c - price) / vega
        if sigma <= 0:
            return None
        if abs(c - price) < tol:
            return sigma
    return None

S, r, T = 100.0, 0.05, 1.0
strikes = [80, 85, 90, 95, 100, 105, 110, 115, 120]
N, STEPS = 500_000, 252
print('Setup complete.')

## Section 1 — Why Black-Scholes Fails: The Volatility Smile

Black-Scholes prices options assuming **constant** volatility $\sigma$.  
If you price a set of options with the same $\sigma$ and then invert each price back
to implied volatility, you trivially recover the same flat $\sigma$ for every strike.

In real markets, implied vols vary with strike — the "smile" or "skew".  The BS model
cannot reproduce this structure.

In [ ]:
sigma_bs = 0.20
bs_ivs   = []

for K in strikes:
    price = mc_engine.GBMPricer(
        spot=S, strike=float(K), rate=r, volatility=sigma_bs, maturity=T,
        n_paths=N, n_steps=STEPS, use_gpu=True).price_european_call().price
    iv = implied_vol_nr(price, S, float(K), r, T, sigma0=sigma_bs)
    bs_ivs.append(iv if iv is not None else float('nan'))
    print(f'  K={K:>3}: MC price={price:.4f}, IV={iv:.4f}')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(strikes, [v * 100 for v in bs_ivs], 'b-o', linewidth=2, markersize=6)
ax.set_xlabel('Strike', fontsize=13)
ax.set_ylabel('Implied Volatility (%)', fontsize=13)
ax.set_title('GBM (Black-Scholes): Flat Implied Volatility Surface', fontsize=13)
ax.set_ylim(15, 25)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print('\nFlat at the input vol — BS trivially recovers σ=0.20 for every strike.')

## Section 2 — The Heston Stochastic Volatility Model

Heston (1993) introduced a two-factor model where variance $v_t$ follows its own SDE:

$$dS_t = \mu S_t\,dt + \sqrt{v_t}\,S_t\,dW_1$$
$$dv_t = \kappa(\theta - v_t)\,dt + \xi\sqrt{v_t}\,dW_2$$

with $\text{Corr}(dW_1, dW_2) = \rho\,dt$.

| Parameter | Symbol | Intuition |
|-----------|--------|----------|
| Initial variance | $v_0$ | Variance at $t=0$ (square of initial vol) |
| Mean reversion speed | $\kappa$ | How quickly $v$ reverts to $\theta$; larger = faster |
| Long-run variance | $\theta$ | Where variance converges in the long run |
| Vol of vol | $\xi$ | How noisy the variance process is; larger = fatter smile |
| Correlation | $\rho$ | Price-vol correlation; **negative $\rho$** → downward skew |

**Why negative $\rho$ produces a skew**: when $\rho < 0$, a falling stock price is
correlated with rising volatility (the "leverage effect").  This makes OTM puts more
expensive (higher IV) and OTM calls cheaper (lower IV) than BS would predict.

## Section 3 — Heston Produces a Smile

Using base parameters $v_0=\theta=0.04$ (i.e. $\sigma_{\text{ATM}}=0.20$),
$\kappa=2$, $\xi=0.3$, $\rho=-0.7$.

In [ ]:
heston_ivs = []
for K in strikes:
    price = mc_engine.HestonPricer(
        spot=S, strike=float(K), rate=r,
        v0=0.04, kappa=2.0, theta=0.04, xi=0.3, rho=-0.7,
        maturity=T, n_paths=N, n_steps=STEPS, use_gpu=True
    ).price_european_call().price
    iv = implied_vol_nr(price, S, float(K), r, T, sigma0=0.20)
    heston_ivs.append(iv if iv is not None else float('nan'))
    print(f'  K={K:>3}: Heston price={price:.4f}, IV={iv:.4f if iv else None}')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(strikes, [v * 100 for v in bs_ivs],     'b-o', linewidth=2, markersize=6,
        label='Black-Scholes (flat)')
ax.plot(strikes, [v * 100 for v in heston_ivs], 'r-s', linewidth=2, markersize=6,
        label='Heston SV (ρ = −0.7)')
ax.set_xlabel('Strike', fontsize=13)
ax.set_ylabel('Implied Volatility (%)', fontsize=13)
ax.set_title('BS vs Heston: Implied Volatility Surface', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print('\nHeston produces a downward skew: OTM puts (low K) have higher IV.')

## Section 4 — Parameter Sensitivity

### (a) Correlation ρ

$\rho$ controls the **direction and steepness** of the skew:
- $\rho < 0$: downward skew (equity markets)
- $\rho = 0$: symmetric smile
- $\rho > 0$: upward skew (commodity markets)

In [ ]:
rho_values = [-0.9, -0.7, -0.3, 0.0, 0.3]

fig, ax = plt.subplots(figsize=(10, 6))
for rho_val in rho_values:
    ivs = []
    for K in strikes:
        price = mc_engine.HestonPricer(
            spot=S, strike=float(K), rate=r,
            v0=0.04, kappa=2.0, theta=0.04, xi=0.3, rho=rho_val,
            maturity=T, n_paths=N, n_steps=STEPS, use_gpu=True
        ).price_european_call().price
        iv = implied_vol_nr(price, S, float(K), r, T, sigma0=0.20)
        ivs.append(iv * 100 if iv is not None else float('nan'))
    ax.plot(strikes, ivs, marker='o', markersize=4, linewidth=1.8, label=f'ρ = {rho_val}')

ax.set_xlabel('Strike', fontsize=13)
ax.set_ylabel('Implied Volatility (%)', fontsize=13)
ax.set_title('Heston Smile: Varying Correlation ρ\n(κ=2, ξ=0.3, v₀=θ=0.04)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### (b) Vol-of-vol ξ

$\xi$ (xi) controls the **curvature** of the smile.  Higher $\xi$ means variance is
more uncertain, making both OTM puts and calls more expensive — widening the smile.

In [ ]:
xi_values = [0.1, 0.2, 0.3, 0.5]

fig, ax = plt.subplots(figsize=(10, 6))
for xi_val in xi_values:
    ivs = []
    for K in strikes:
        price = mc_engine.HestonPricer(
            spot=S, strike=float(K), rate=r,
            v0=0.04, kappa=2.0, theta=0.04, xi=xi_val, rho=-0.7,
            maturity=T, n_paths=N, n_steps=STEPS, use_gpu=True
        ).price_european_call().price
        iv = implied_vol_nr(price, S, float(K), r, T, sigma0=0.20)
        ivs.append(iv * 100 if iv is not None else float('nan'))
    ax.plot(strikes, ivs, marker='o', markersize=4, linewidth=1.8, label=f'ξ = {xi_val}')

ax.set_xlabel('Strike', fontsize=13)
ax.set_ylabel('Implied Volatility (%)', fontsize=13)
ax.set_title('Heston Smile: Varying Vol-of-Vol ξ\n(κ=2, ρ=−0.7, v₀=θ=0.04)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### (c) Mean reversion κ

$\kappa$ (kappa) controls how quickly the variance reverts to $\theta$.  
Higher $\kappa$ = faster mean reversion = variance stays closer to $\theta$
= flatter smile (less room for large vol excursions).

In [ ]:
kappa_values = [0.5, 1.0, 2.0, 5.0]

fig, ax = plt.subplots(figsize=(10, 6))
for kappa_val in kappa_values:
    ivs = []
    for K in strikes:
        price = mc_engine.HestonPricer(
            spot=S, strike=float(K), rate=r,
            v0=0.04, kappa=kappa_val, theta=0.04, xi=0.3, rho=-0.7,
            maturity=T, n_paths=N, n_steps=STEPS, use_gpu=True
        ).price_european_call().price
        iv = implied_vol_nr(price, S, float(K), r, T, sigma0=0.20)
        ivs.append(iv * 100 if iv is not None else float('nan'))
    ax.plot(strikes, ivs, marker='o', markersize=4, linewidth=1.8, label=f'κ = {kappa_val}')

ax.set_xlabel('Strike', fontsize=13)
ax.set_ylabel('Implied Volatility (%)', fontsize=13)
ax.set_title('Heston Smile: Varying Mean Reversion κ\n(ξ=0.3, ρ=−0.7, v₀=θ=0.04)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Section 5 — Heston vs Black-Scholes: When Does It Matter?

For **long maturities** mean reversion pulls variance back to $\theta$, so the two
models agree more.  For **short maturities** there is less time for mean reversion,
and the stochastic variance has a bigger effect on the terminal distribution.

In [ ]:
maturities = [0.1, 0.25, 0.5, 1.0, 2.0]
Katm       = 100.0   # ATM

bs_prices, heston_prices = [], []

for Ti in maturities:
    p_bs = mc_engine.GBMPricer(
        spot=S, strike=Katm, rate=r, volatility=0.20, maturity=Ti,
        n_paths=N, n_steps=max(int(252 * Ti), 10), use_gpu=True
    ).price_european_call().price

    p_h = mc_engine.HestonPricer(
        spot=S, strike=Katm, rate=r,
        v0=0.04, kappa=2.0, theta=0.04, xi=0.3, rho=-0.7, maturity=Ti,
        n_paths=N, n_steps=max(int(252 * Ti), 10), use_gpu=True
    ).price_european_call().price

    bs_prices.append(p_bs)
    heston_prices.append(p_h)
    print(f'  T={Ti:.2f}: BS={p_bs:.4f}, Heston={p_h:.4f}, diff={p_h-p_bs:+.4f}')

diffs = [h - b for h, b in zip(heston_prices, bs_prices)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(maturities, bs_prices,     'b-o', linewidth=2, markersize=6, label='Black-Scholes')
axes[0].plot(maturities, heston_prices, 'r-s', linewidth=2, markersize=6, label='Heston')
axes[0].set_xlabel('Maturity T (years)', fontsize=12)
axes[0].set_ylabel('ATM Call Price', fontsize=12)
axes[0].set_title('ATM Call Price vs Maturity', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].bar(maturities, diffs, color=['green' if d >= 0 else 'red' for d in diffs],
            width=0.07, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Maturity T (years)', fontsize=12)
axes[1].set_ylabel('Heston − BS price', fontsize=12)
axes[1].set_title('Price Difference: Heston − Black-Scholes', fontsize=12)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()